# TRABAJO BIG DATA

In [146]:
# importamos paquetes
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import numpy as np
import pandas as pd
import os
import time
from datetime import datetime

In [148]:
# directorio
# ruta absoluta del original, no portable:
# os.chdir('/Users/catalinagallardo/Downloads/University/Big Data/Trabajo')

In [150]:
# NOTA: las credenciales fueron removidas de esta copia archivada.
# El pipeline corregido las lee de variables de entorno;
# ver programs/01_construir_base.py y el README.
# credenciales API
CLIENT_ID1 = '<REDACTADO>'
CLIENT_SECRET1 = '<REDACTADO>'

In [152]:
# NOTA: las credenciales fueron removidas de esta copia archivada.
# El pipeline corregido las lee de variables de entorno;
# ver programs/01_construir_base.py y el README.
# credenciales API 
CLIENT_ID2 = "<REDACTADO>"
CLIENT_SECRET2 = "<REDACTADO>"

## Armamos la base de datos

In [155]:
# importamos bases de datos
df_2018 = pd.read_csv('data_2018.csv')
df_2019 = pd.read_csv('data_2019.csv')

In [156]:
# renombramos columnas para facilitar análisis
df_2018.rename(columns={'popularity': 'popularity_2018'}, inplace=True)
df_2019.rename(columns={'popularity': 'popularity_2019'}, inplace=True)

In [157]:
# eliminamos duplicados
df_2018 = df_2018.drop_duplicates(subset='track_id')
df_2019 = df_2019.drop_duplicates(subset='track_id')

In [158]:
# lista de canciones de ambas bases
df = pd.merge(df_2019['track_id'], df_2018['track_id'], on='track_id', how='outer')
track_ids = df['track_id'].tolist()

In [165]:
# creamos listas y diccionarios para poder iterar
song_data = {'track_id': [], 'artist_id': [], 'album_id': [], 'album_name': [], 'date': [], 'album_type': [],
             'track_number': [], 'explicit': [], 'artist_count': []}

In [167]:
# tomamos información de las canciones buscando en la API
i = 0
n = 0
batch_size = 50

for x in range(0, len(track_ids), batch_size):
    batch = track_ids[x:x + batch_size]
    
    if n == 0:
        sp1 = spotipy.Spotify(auth_manager=SpotifyClientCredentials(client_id=CLIENT_ID1, client_secret=CLIENT_SECRET1), requests_timeout=100)
        results = sp1.tracks(batch)['tracks']
    elif n < 20:
        results = sp1.tracks(batch)['tracks']
    elif n == 20:
        sp2 = spotipy.Spotify(auth_manager=SpotifyClientCredentials(client_id=CLIENT_ID2, client_secret=CLIENT_SECRET2), requests_timeout=100)
        results = sp2.tracks(batch)['tracks']
    elif n < 40:
        results = sp2.tracks(batch)['tracks']
    else:
        results = sp2.tracks(batch)['tracks']
        n = 0

    for track in results:
        if track['album']['release_date'] == '0000':
            continue
        
        song_data['track_id'].append(track['id'])
        song_data['artist_id'].append(track['artists'][0]['id'])
        song_data['album_id'].append(track['album']['id'])
        song_data['album_name'].append(track['album']['name'])
        song_data['date'].append(track['album']['release_date'])
        song_data['album_type'].append(track['album']['album_type'])
        song_data['track_number'].append(track['track_number'])
        song_data['explicit'].append(track['explicit'])
        song_data['artist_count'].append(len(track['artists']))
    
    i += 1
    
    if i > 10:
        time.sleep(5)
        n += 1
        i = 0

In [168]:
# agregamos la información extra de las canciones a las bases de datos 
info = pd.DataFrame(song_data)
df_2018 = df_2018.merge(info, on='track_id', how='inner')
df_2019 = df_2019.merge(info, on='track_id', how='inner')

## Limpiamos los datos

In [170]:
# comvertimos la variale explicit a dummy
df_2018['explicit_dummy'] = df_2018['explicit'].astype(int)
df_2019['explicit_dummy'] = df_2019['explicit'].astype(int)

In [183]:
# borramos observaciones de canciones que ya no existen en spotify
df_2018['date'] = pd.to_datetime(df_2018['date'], errors='coerce')
df_2018 = df_2018[df_2018['date'] != 0000]

df_2019['date'] = pd.to_datetime(df_2019['date'], errors='coerce')
df_2019 = df_2019[df_2019['date'] != 0000]

In [185]:
# calculamos edad de la cancion
df_2018['song_age'] = 2018 - df_2018['date'].dt.year
df_2019['song_age'] = 2019 - df_2019['date'].dt.year

In [187]:
# corrección: eliminamos canciones con fecha más nueva que 2018/2019 (son reversiones) para no sesgar analisis
df_2018 = df_2018[df_2018['date'].dt.year <= 2018]
df_2019 = df_2019[df_2019['date'].dt.year <= 2019]

## Descargamos bases completas

In [189]:
# exportamos a csv
df_2018.to_csv('base_2018.csv', index=False)
df_2019.to_csv('base_2019.csv', index=False)

FIN :)